In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook visualizes ALL 2N poles of the function
#
#       H(s)H(-s)
#
# associated with a continuous-time Chebyshev Type I low-pass filter.
#
# Three parameters can be varied:
#
#       N     : filter order
#       ωp    : passband-edge angular frequency
#       Ap    : maximum passband ripple in dB
#
# The passband-ripple parameter is
#
#       ε = sqrt(10^(Ap/10) - 1)
#
# The 2N poles lie on an ellipse with semi-axes
#
#       α = ωp sinh[(1/N) asinh(1/ε)]
#       β = ωp cosh[(1/N) asinh(1/ε)]
#
# where α is the horizontal semi-axis and β is the vertical semi-axis.
#
# Since
#
#       β² - α² = ωp²
#
# the focal distance is
#
#       c = sqrt(β² - α²) = ωp
#
# and therefore the two foci are located at
#
#       F1 = -jωp
#       F2 = +jωp
#
# Pole representation:
#
#       Filled red circles : poles used in the stable transfer function H(s)
#       Open red circles   : poles rejected because Re{p} > 0
#       Green circles      : foci of the Chebyshev ellipse
#
# ==============================================================================

# ==============================================================================
# DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 8px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:1040px;
    max-width:1040px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Visualize all 2N poles of H(s)H(-s) for a Chebyshev Type I low-pass filter and identify the N poles that form the stable transfer function H(s).
<br>
<b>Interpretation:</b>
The poles lie on an ellipse whose horizontal and vertical semi-axes depend on N, ω<sub>p</sub> and A<sub>p</sub>. Filled red circles correspond to the N poles in the left half-plane that are retained in H(s), while open red circles correspond to the N right-half-plane poles that are rejected. Changing A<sub>p</sub> or N modifies the geometry of the ellipse. For constant ω<sub>p</sub>, however, β² − α² = ω<sub>p</sub>², so the two foci remain fixed at ±jω<sub>p</sub> while the ellipse changes shape.
</div>
""", layout=Layout(width='1050px', max_width='1050px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='280px')
style_opts = {'description_width':'90px'}

order_slider = IntSlider(min=2, max=10, step=1, value=5, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)

wp_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=5.0, description='ωp:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

Ap_slider = FloatSlider(min=0.5, max=3.0, step=0.1, value=1.0, description='Ap (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='350px', max_width='350px'))

pole_table = HTML(layout=Layout(width='450px', max_width='450px'))

# ==============================================================================
# FIGURE
# ==============================================================================

fig, ax = plt.subplots(figsize=(6.6, 6.6))

ellipse_parameter = np.linspace(0.0, 2.0 * np.pi, 900)

ellipse_line, = ax.plot([], [], 'r--', linewidth=1.3, alpha=0.65, label='Chebyshev ellipse')

used_scatter = ax.scatter([], [], s=95, marker='o', facecolors='red', edgecolors='red', linewidths=1.5, label='Used poles')

rejected_scatter = ax.scatter([], [], s=95, marker='o', facecolors='white', edgecolors='red', linewidths=1.8, label='Rejected poles')

foci_scatter = ax.scatter([], [], s=75, marker='o', facecolors='green', edgecolors='green', linewidths=1.4, label='Ellipse foci')

ax.axhline(0.0, color='black', linewidth=0.9)
ax.axvline(0.0, color='black', linewidth=0.9)

ax.set_xlabel('Re{s}', fontsize=11)
ax.set_ylabel('Im{s}', fontsize=11)

ax.set_title('Chebyshev I Poles of H(s)H(-s)', fontsize=13, fontweight='bold', pad=8)

ax.grid(True, linestyle=':', alpha=0.35)

ax.set_aspect('equal', adjustable='box')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=4, fontsize=8)

axis_limit = 8.0

ax.set_xlim(-axis_limit, axis_limit)
ax.set_ylim(-axis_limit, axis_limit)

ax.set_xticks(np.arange(-8, 9, 2))
ax.set_yticks(np.arange(-8, 9, 2))

fig.subplots_adjust(left=0.12, right=0.96, bottom=0.17, top=0.92)

fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.resizable = False

fig.canvas.layout.width = '660px'
fig.canvas.layout.height = '660px'

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_chebyshev1_poles(change=None):

    N = order_slider.value
    wp = wp_slider.value
    Ap = Ap_slider.value

    # --------------------------------------------------------------------------
    # Ripple parameter
    # --------------------------------------------------------------------------

    epsilon = np.sqrt(10.0**(Ap / 10.0) - 1.0)

    # --------------------------------------------------------------------------
    # Chebyshev ellipse semi-axes
    # --------------------------------------------------------------------------

    mu = np.arcsinh(1.0 / epsilon) / N

    alpha = wp * np.sinh(mu)
    beta = wp * np.cosh(mu)

    # --------------------------------------------------------------------------
    # Focal distance
    #
    # β² - α² = ωp²
    #
    # Therefore:
    #
    # c = sqrt(β² - α²) = ωp
    #
    # Since the major axis is vertical, the foci are located at
    #
    # F1 = -jωp
    # F2 = +jωp
    # --------------------------------------------------------------------------

    focal_distance = np.sqrt(beta**2 - alpha**2)

    foci = np.array([0.0 - 1j * focal_distance, 0.0 + 1j * focal_distance])

    # --------------------------------------------------------------------------
    # Compute ALL 2N poles of H(s)H(-s)
    # --------------------------------------------------------------------------

    k = np.arange(1, 2 * N + 1)

    angles = (2 * k - 1) * np.pi / (2 * N)

    sigma = alpha * np.sin(angles)
    omega_poles = beta * np.cos(angles)

    poles = sigma + 1j * omega_poles

    # --------------------------------------------------------------------------
    # Stable and rejected poles
    # --------------------------------------------------------------------------

    tolerance = 1e-12

    used_mask = np.real(poles) < -tolerance
    rejected_mask = np.real(poles) > tolerance

    used_poles = poles[used_mask]
    rejected_poles = poles[rejected_mask]

    # --------------------------------------------------------------------------
    # Update ellipse
    # --------------------------------------------------------------------------

    ellipse_x = alpha * np.cos(ellipse_parameter)
    ellipse_y = beta * np.sin(ellipse_parameter)

    ellipse_line.set_data(ellipse_x, ellipse_y)

    # --------------------------------------------------------------------------
    # Update used poles
    # --------------------------------------------------------------------------

    if len(used_poles) > 0:
        used_offsets = np.column_stack((np.real(used_poles), np.imag(used_poles)))
    else:
        used_offsets = np.empty((0, 2))

    used_scatter.set_offsets(used_offsets)

    # --------------------------------------------------------------------------
    # Update rejected poles
    # --------------------------------------------------------------------------

    if len(rejected_poles) > 0:
        rejected_offsets = np.column_stack((np.real(rejected_poles), np.imag(rejected_poles)))
    else:
        rejected_offsets = np.empty((0, 2))

    rejected_scatter.set_offsets(rejected_offsets)

    # --------------------------------------------------------------------------
    # Update ellipse foci
    # --------------------------------------------------------------------------

    foci_offsets = np.column_stack((np.real(foci), np.imag(foci)))

    foci_scatter.set_offsets(foci_offsets)

    # --------------------------------------------------------------------------
    # Pole table
    # --------------------------------------------------------------------------

    rows = ""

    for index, p in enumerate(poles):

        if np.real(p) < -tolerance:

            status = "USED"

            status_style = """
                color:#0066cc;
                background:#eef6ff;
                border:1px solid #9bc8f5;
            """

        elif np.real(p) > tolerance:

            status = "REJECTED"

            status_style = """
                color:#cc0000;
                background:#fff1f1;
                border:1px solid #efaaaa;
            """

        else:

            status = "BOUNDARY"

            status_style = """
                color:#666666;
                background:#f2f2f2;
                border:1px solid #cccccc;
            """

        rows += f"""
        <tr style="border-bottom:1px solid #eeeeee;">

            <td style="
                padding:5px 8px;
                text-align:center;
                font-family:'Times New Roman',serif;
                font-size:17px;
                font-style:italic;
                white-space:nowrap;
            ">
                p<sub>{index}</sub>
            </td>

            <td style="
                padding:5px 10px;
                font-family:'Times New Roman',serif;
                font-size:16px;
                white-space:nowrap;
            ">
                {p.real:+.6f}
                <span style="font-style:italic;">{p.imag:+.6f}j</span>
            </td>

            <td style="
                padding:5px 8px;
                text-align:center;
            ">
                <span style="
                    {status_style}
                    padding:2px 7px;
                    border-radius:10px;
                    font-size:10px;
                    font-weight:bold;
                    letter-spacing:0.3px;
                    white-space:nowrap;
                ">
                    {status}
                </span>
            </td>

        </tr>
        """

    pole_table.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px;
        background:white;
        width:440px;
        max-height:620px;
        overflow-y:auto;
        box-sizing:border-box;
        font-size:12px;
    ">

    <div style="
        font-family:'Times New Roman',serif;
        font-size:18px;
        font-weight:bold;
        margin-bottom:7px;
        text-align:center;
    ">
        Pole Values
    </div>

    <table style="
        width:100%;
        border-collapse:collapse;
    ">

        <tr style="border-bottom:1px solid #bbbbbb;">
            <th style="padding:5px;">Pole</th>
            <th style="padding:5px;">Complex Value</th>
            <th style="padding:5px;">Status</th>
        </tr>

        {rows}

    </table>

    </div>
    """

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    if N % 2 == 1:

        parity_text = "Odd order"

        stable_real_poles = used_poles[np.abs(np.imag(used_poles)) < 1e-10]

        if len(stable_real_poles) > 0:
            real_pole_text = f"One stable real pole at s = {stable_real_poles[0].real:.4f}"
        else:
            real_pole_text = "One stable real pole"

    else:

        parity_text = "Even order"
        real_pole_text = "No real pole"

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 10px;
        margin-top:8px;
        font-size:12px;
        line-height:1.65;
        background:white;
        width:345px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Chebyshev Type I low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Passband edge:</b>
        <span style="color:#0066cc;">ωp = {wp:.2f} rad/s</span>
    </div>

    <div>
        <b>Passband ripple:</b>
        <span style="color:#0066cc;">Ap = {Ap:.2f} dB</span>
    </div>

    <div>
        <b>Ripple parameter:</b>
        <span style="color:#0066cc;">ε = {epsilon:.6f}</span>
    </div>

    <div style="
        margin-top:5px;
        padding-top:5px;
        border-top:1px solid #eeeeee;
    ">
        <b>Ellipse geometry:</b>
    </div>

    <div>
        <b>Horizontal semi-axis α:</b>
        <span style="color:#0066cc;">{alpha:.6f}</span>
    </div>

    <div>
        <b>Vertical semi-axis β:</b>
        <span style="color:#0066cc;">{beta:.6f}</span>
    </div>

    <div>
        <b>Axis ratio β/α:</b>
        <span style="color:#0066cc;">{beta / alpha:.4f}</span>
    </div>

    <div>
        <b>Focal relation:</b>
        <span style="color:#008000;">β² − α² = ωp² = {wp**2:.6f}</span>
    </div>

    <div>
        <b>Focal distance c:</b>
        <span style="color:#008000;">{focal_distance:.6f}</span>
    </div>

    <div>
        <b>Upper focus F₁:</b>
        <span style="color:#008000;">+j{focal_distance:.6f}</span>
    </div>

    <div>
        <b>Lower focus F₂:</b>
        <span style="color:#008000;">−j{focal_distance:.6f}</span>
    </div>

    <div style="
        margin-top:5px;
        padding-top:5px;
        border-top:1px solid #eeeeee;
    ">
        <b>Total poles:</b>
        <span style="color:#0066cc;">{2 * N}</span>
    </div>

    <div>
        <b>Used poles:</b>
        <span style="color:#0066cc;">{len(used_poles)}</span>
    </div>

    <div>
        <b>Rejected poles:</b>
        <span style="color:#0066cc;">{len(rejected_poles)}</span>
    </div>

    <div>
        <b>Order type:</b>
        <span style="color:#0066cc;">{parity_text}</span>
    </div>

    <div>
        <b>Real pole:</b>
        <span style="color:#0066cc;">{real_pole_text}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        All 2N poles lie on a Chebyshev ellipse. For constant ωp,
        changing N or Ap changes both semi-axes and therefore deforms
        the ellipse, but β² − α² = ωp² remains constant. Consequently,
        the two foci remain fixed at ±jωp while the poles move along
        the changing ellipse. Only the N poles in the left half-plane
        are retained to construct the stable transfer function H(s).
    </div>

    </div>
    """

    fig.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_chebyshev1_poles, names='value')
wp_slider.observe(update_chebyshev1_poles, names='value')
Ap_slider.observe(update_chebyshev1_poles, names='value')

# ==============================================================================
# LAYOUT
# ==============================================================================

controls = VBox([parameter_title, order_slider, wp_slider, Ap_slider, info_html], layout=Layout(width='360px', min_width='360px', max_width='360px', flex='0 0 360px', align_items='flex-start'))

main_row = HBox([controls, fig.canvas, pole_table], layout=Layout(width='1480px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_chebyshev1_poles()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_row)